# Threat Detection Quickstart

Connect to Clario 360, pull recent alerts, visualize severity patterns, and export a report.

In [ ]:
from datetime import datetime, timedelta
import pandas as pd
import plotly.express as px
from clario360 import Client

DAYS_BACK = int(__import__('os').environ.get('CLARIO360_DAYS_BACK', '7'))
SEVERITIES = 'critical,high'
sdk = Client.from_env()
identity = sdk.whoami()
print(f"Connected as {identity.email} in tenant {identity.tenant_name}")

## Pull alerts

The SDK automatically authenticates with the notebook session token injected by JupyterHub.

In [ ]:
try:
    alerts = sdk.cyber.alerts.list(
        severity=SEVERITIES,
        date_from=(datetime.utcnow() - timedelta(days=DAYS_BACK)).isoformat(),
        per_page=100,
        sort='created_at',
        order='desc',
    )
    alerts_df = alerts.to_dataframe()
    print(f"Loaded {len(alerts_df)} alerts")
except Exception as exc:
    raise RuntimeError(f'Failed to load alerts: {exc}') from exc

alerts_df.head(10)

## Visualize and export

These cells are safe to rerun. They only read from the platform and write a CSV into your home directory.

In [ ]:
if alerts_df.empty:
    print('No alerts found for the selected window.')
else:
    counts = alerts_df['severity'].value_counts()
    fig = px.pie(values=counts.values, names=counts.index, title='Alert Severity Distribution')
    fig.show()
    alerts_df['hour'] = pd.to_datetime(alerts_df['created_at']).dt.floor('h')
    timeline = alerts_df.groupby(['hour', 'severity']).size().reset_index(name='count')
    fig2 = px.line(timeline, x='hour', y='count', color='severity', title='Alert Volume Over Time')
    fig2.show()
    output_file = f"critical_alerts_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.csv"
    alerts_df.to_csv(output_file, index=False)
    print(f'Exported {len(alerts_df)} rows to {output_file}')